# Tutorial 2: Ray Real-Time Streaming Pipeline

This notebook demonstrates how to connect to a Ray cluster to run ML jobs, and specifically how to simulate a real-time data ingestion and processing pipeline in a streaming fashion.

## Part 1: Connecting to Ray
Initialize Ray to connect to a local or remote cluster.

In [ ]:
import ray
import xarray as xr
import time
from panoseti_analysis.adapters.ray.launcher import init_ray

# Connect to Ray
init_ray(launcher="attach")
print("Ray connected!", ray.cluster_resources())

## Part 2: Streaming Data Processing

We can simulate data arriving in a streaming fashion (e.g. from an observatory) and process it asynchronously using Ray remote functions.

In [ ]:
@ray.remote
def process_data_chunk(chunk_id, data_chunk):
    """Simulate processing a single chunk of data asynchronously."""
    # Simulate compute time
    time.sleep(1)
    # Perform dummy calculation
    mean_val = float(data_chunk.mean())
    return f"Processed chunk {chunk_id} | Mean: {mean_val:.3f}"

In [ ]:
# Let's load the test L1 store to act as our data source
l1_path = "results_tutorial/L1/obs_TEST.dp_img16.module_1.L1.zarr"
ds = xr.open_zarr(l1_path, consolidated=False)
img_data = ds["median_subtracted"].values

# Simulate streaming 5 frames of data
futures = []
for i in range(5):
    print(f"Stream received frame {i}...")
    chunk = img_data[i]
    # Dispatch to Ray cluster asynchronously
    fut = process_data_chunk.remote(i, chunk)
    futures.append(fut)

# Wait for all real-time processing to finish
results = ray.get(futures)
for res in results:
    print(res)